# exp-20260914-12 — our own LLM label key

The public key (`llm_labels_v4_blend`, 0.8927 on gold) is CC0 and the host explicitly permits
LLM-derived labels. But it names no model and publishes no prompt, so we cannot reproduce, correct
or extend it. This builds a key we own: **`Qwen2.5-7B-Instruct`, greedy decoding, one report per
call, prompt in `labels/prompt-v1.md`** — all offline, inside the competition notebook.

Two reasons this is worth doing even if it scores below 0.8927:

1. **Auditability.** The winners' obligation is to open-source a working solution. A pipeline whose
   targets come from an unreproducible third-party CSV is publishable but not reproducible.
2. **Decorrelation pays more than quality here.** The public blend gained most from its *weakest*
   partner (0.8347 alone, rank-correlation 0.718) because it was the least correlated. An
   independent key is worth having even at a lower solo score.

**Gold discipline:** 29 dev / 29 test, fixed seed. Prompt revisions may look only at dev; every
number is reported on test. Stated openly because 58 studies leave no cleaner option.

**This run is the PILOT** — `N_PILOT` reports, enough to measure per-report latency and score the
prompt, before committing to all 4,407.


In [ ]:
# ================================================================== CONFIG — the only cell to edit
from __future__ import annotations
import glob, json, os, re, time, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd

RUN_MODE = "full"          # "smoke" (minutes, proves it runs) | "full" (the baseline) | "submit" (inference only)

# ---- competition constants (rules.md) -----------------------------------------------------------
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_LABEL = len(TARGETS)
SUBMISSION_NAME = "submission.csv"        # rules.md: hard requirement
KAGGLE_LIMIT_H  = 9.0                     # rules.md: CPU or GPU notebook <= 9 h
WORKING_LIMIT_H = 6.75                    # rules.md: 9 h minus 25% headroom

# ---- versioned artefacts (rules.md: cache is keyed by preprocessing version) --------------------
LABELLER_VERSION  = "v1-keyword"          # weak-label rules; bump when the labeller changes
PREPROC_VERSION   = "p1"                  # bump on ANY change to the DICOM -> tensor path
EXPERIMENT_ID     = "exp-12-own-llm-labels"

# ---- study -> tensor geometry -------------------------------------------------------------------
SLOTS       = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1)]   # (plane, prefer fluid-sensitive)
N_SLOT      = len(SLOTS)
N_TRIPLET   = 4            # windows per slot; a window = 3 adjacent slices stacked as channels
IMG         = 192
CROP_MM     = 130.0
SLICE_BAND  = (0.15, 0.85)
K           = N_SLOT * N_TRIPLET

# ---- model / training ---------------------------------------------------------------------------
BACKBONE     = "resnet18"
PRETRAINED   = True        # with internet off this needs an attached weights dataset; see below
EPOCHS       = 4           # FIXED. Never chosen by looking at gold (rules.md hard rule 2)
BATCH        = 8
LR_HEAD      = 3e-4
LR_BACKBONE  = 1e-4
NUM_WORKERS  = 2

# ---- evaluation protocol (rules.md: statistical rules) ------------------------------------------
N_SEEDS   = {"smoke": 1, "full": 3, "submit": 1}[RUN_MODE]   # training seeds -> seed variance
SEEDS     = [2026, 2027, 2028][:N_SEEDS]
N_FOLDS   = 5              # evaluation folds over the 58 gold studies (NOT training folds)
EVAL_REPEATS = 5           # fold reshuffles; sigma comes from (seed x repeat x fold) cells
# Measured on a T4: with a warm cache an epoch costs ~2 s per 120 studies, so training is
# decode-bound, not compute-bound. With the prebuilt cache attached, use every weakly-labelled
# study — the old 1200 cap only ever existed to fit a decode budget the cache removes.
MAX_TRAIN_STUDIES = {"smoke": 120, "full": 10_000, "submit": 0}[RUN_MODE]
# Prebuilt tensor cache: the output of notebooks/cache-build-p1.ipynb, attached as a data source.
# Kaggle mounts a notebook's output at /kaggle/input/<notebook-slug>/ (plus our cache_<ver> subdir).
CACHE_INPUT_DIRS  = ["/kaggle/input/rsna-knee-cache-build-p1/cache_p1",
                     "/kaggle/input/rsna-knee-cache-build-p1",
                     "/kaggle/input/rsna-knee-cache-p1"]

# ---- offline weights (rules.md: internet is disabled in the rerun) ------------------------------
# Backbone weights ship as an attached dataset because the rerun cannot download anything.
# Dataset: kaggle.com/datasets/vaibhav486/timm-backbones-offline (Apache-2.0, redistributable —
# required by the winners' obligation to publish weights).
TIMM_OFFLINE_DIRS = ["/kaggle/input/timm-backbones-offline", "/kaggle/input/timm-weights"]
BACKBONE_WEIGHTS = {                      # timm model name -> file in the dataset above
    "resnet18":           "resnet18.a1_in1k.bin",
    "resnet34":           "resnet34.a1_in1k.bin",
    "tf_efficientnet_b0": "tf_efficientnet_b0.ns_jft_in1k.bin",
    "convnext_tiny":      "convnext_tiny.in12k_ft_in1k.bin",
}
CHECKPOINT_DIRS   = ["/kaggle/input/rsna-knee-baseline-v1"]   # our own trained weights, for "submit"

# ---- paths ---------------------------------------------------------------------------------------
def find_root() -> Path:
    for c in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"), Path(".")]:
        if (c / "train.csv").exists() or list(c.glob("train*.csv")):
            return c
    raise FileNotFoundError("Competition data not found; set ROOT by hand.")

def find_csv(root: Path, stem: str) -> Path:
    exact = root / f"{stem}.csv"
    if exact.exists():
        return exact
    hits = sorted(c for c in root.glob(f"{stem}*.csv") if "_series" not in c.name)
    if not hits:
        raise FileNotFoundError(f"{stem}.csv not found under {root}")
    return hits[0]

ROOT  = find_root()
WORK  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / f"cache_{PREPROC_VERSION}"          # version in the path: a stale cache cannot be reused
CACHE.mkdir(parents=True, exist_ok=True)


def discover_input(pattern: str, want_dir: bool = False) -> list[Path]:
    """Find something under /kaggle/input without guessing the mount layout.

    Kaggle has mounted attachments at BOTH /kaggle/input/<slug> and the nested
    /kaggle/input/{datasets,notebooks,competitions}/<owner>/<slug>/[version]/ — and which one you
    get is not under our control. Hardcoding either cost a wasted GPU hour once already, so search
    instead and print what was found."""
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    hits = [p for p in sorted(root.glob(pattern)) if (p.is_dir() if want_dir else p.is_file())]
    return hits


# A prebuilt cache (attached notebook output) is searched first, then our own writable one.
_cache_hits = [Path(d) for d in CACHE_INPUT_DIRS if Path(d).exists()]
_cache_hits += [d for d in discover_input(f"**/cache_{PREPROC_VERSION}", want_dir=True)
                if d not in _cache_hits and d != CACHE]
CACHE_READ = _cache_hits + [CACHE]

T_START = time.time()
def elapsed_h() -> float:
    return (time.time() - T_START) / 3600.0

def budget_check(stage: str) -> None:
    """rules.md hard rule 7: stay inside the 6.75 h working limit, loudly."""
    h = elapsed_h()
    print(f"[budget] {stage}: {h:.2f} h of {WORKING_LIMIT_H} h used ({h / KAGGLE_LIMIT_H:.0%} of the Kaggle cap)")
    if h > WORKING_LIMIT_H:
        warnings.warn(f"OVER THE WORKING BUDGET at '{stage}' — this configuration is not submittable.")

np.random.seed(SEEDS[0])
print(f"run mode   : {RUN_MODE}   seeds={SEEDS}   epochs={EPOCHS}")
print(f"data root  : {ROOT.resolve()}")
print(f"work dir   : {WORK.resolve()}")
print(f"cache      : {CACHE.name}  (preproc {PREPROC_VERSION}, labeller {LABELLER_VERSION})")
print(f"per study  : {K} windows ({N_SLOT} slots x {N_TRIPLET} triplets) at {IMG}x{IMG}")


In [ ]:
test = pd.read_csv(find_csv(ROOT, "test"))
train        = pd.read_csv(find_csv(ROOT, "train"))
train_series = pd.read_csv(find_csv(ROOT, "train_series"))
test_series  = pd.read_csv(find_csv(ROOT, "test_series"))
for df in (train, test, train_series, test_series):
    for col in ("StudyInstanceUID", "SeriesInstanceUID"):
        if col in df.columns:
            df[col] = df[col].astype(str)

gold_mask = train[TARGETS].notna().all(axis=1)
gold = train.loc[gold_mask].reset_index(drop=True)
Y_GOLD = gold[TARGETS].values.astype(int)          # [58, 12] — the only ground truth we own

pos = pd.Series(Y_GOLD.sum(0), index=TARGETS)
print(f"gold studies: {len(gold)}   report-only: {(~gold_mask).sum()}")
print("\npositives per target among the gold studies:")
print(pos.to_string())
print(f"\nrarest target: {pos.idxmin()} with {pos.min()} positives -> "
      f"{pos.min() / N_FOLDS:.1f} expected positives per fold. "
      "This is why undefined (fold, label) cells are unavoidable and must be counted.")


In [ ]:
def normalise(text: str) -> str:
    t = unicodedata.normalize("NFKD", str(text))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", t.lower())

NEG = (r"(?:no |not |without |absence of |negative for |intact |normal |unremarkable |ohne |"
       r"kein[e]?[nrms]? |unauff|sin |ausencia|geen |zonder |normale |normaal |bez |uredn|"
       r"nema |sans |pas de |absence)")

PATTERNS = {
    "ACL": r"(acl|anterior cruciate|lca|vkb|ligamento cruzado anterior|voorste kruisband|"
           r"kruisband anterior|prednj[ei] krizn|kreuzband(?:ruptur)?\s*(?:vorder)?|vorderes kreuzband)",
    "MCL": r"(mcl|medial collateral|ligamento colateral medial|innenband|mediale[nr]? kollateralband|"
           r"mediale collaterale|medijalni kolateralni)",
    "Medial Meniscus": r"(medial meniscus|menisco (?:interno|medial)|innenmeniskus|mediale meniscus|"
                       r"medijalni menisk|meniscus medialis|meniscus internus)",
    "Lateral Meniscus": r"(lateral meniscus|menisco (?:externo|lateral)|aussenmeniskus|laterale meniscus|"
                        r"lateralni menisk|meniscus lateralis)",
    "Medial OA": None, "Lateral OA": None, "PF OA": None,       # compartment co-occurrence, below
    "Effusion": r"(effusion|derrame|gelenkerguss|ergus[s]?|hydrops|izljev|epanchement|joint fluid|"
                r"gewrichtsvocht|vocht)",
    "Synovitis": r"(synovit\w*|sinovit\w*|synovialit\w*|sinovij\w*|"
                 r"synovial\w* (?:proliferation|thickening|verdikking|hypertroph\w*)|"
                 r"proliferacij\w* sinovij|pannus|synoviale? reizung)",
    "Baker's": r"(baker|popliteal cyst|quiste de baker|bakerzyste|baker-zyste|bakerova cist|kyste de baker)",
    "Contusion": r"(bone (?:marrow )?(?:contusion|bruise|oedema|edema)|contusion|knochenmarkod|kontuzij|"
                 r"botcontusie|edema oseo)",
    "Fracture": r"(fracture|fractur|fraktur|fisura osea|prijelom|breuk|fractuur|avulsion)",
}
ABNORMAL = (r"(tear|rupt|riss|scheur|rotura|lesion|desgarr|lasion|laesion|degenerativ|signal|"
            r"tearing|ruptura|insuffizienz|discontinu|abnormal)")
NEEDS_ABNORMAL = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_WORD = (r"(osteoarthrit\w*|arthros\w*|artros\w*|artroz\w*|gonarthros\w*|osteoartr\w*|chondral loss|"
           r"cartilage loss|knorpel\w*|chondropath\w*|kraakbeen\w*|hrskavic\w*|osteophyt\w*|osteofit\w*|"
           r"degenerative (?:change|veranderung)\w*|denudacij\w*)")
COMPARTMENT = {
    "Medial OA":  r"(medial\w*|mediaal|medijaln\w*|intern[oa]|innen\w*|inner)",
    "Lateral OA": r"(lateral\w*|lateraal|lateraln\w*|extern[oa]|aussen\w*|outer)",
    "PF OA":      r"(patell\w*|patel\w*|femoropatel\w*|retropatell\w*|trochlea\w*|trohlej\w*)",
}
WINDOW = 60

def _oa_label(t: str, key: str) -> float:
    pos = neg = 0
    for m in re.finditer(OA_WORD, t):
        s, e = m.span()
        ctx = t[max(0, s - WINDOW):e + WINDOW]
        if not re.search(COMPARTMENT[key], ctx):
            continue
        if key != "PF OA" and re.search(r"(?:patell|patel|trochlea|trohlej)", ctx):
            continue
        if re.search(NEG + r"[^.]{0,25}$", t[max(0, s - 45):s]):
            neg += 1
        else:
            pos += 1
    return 1.0 if pos else (0.0 if neg else np.nan)

def label_report(text: str) -> dict:
    """{finding: 1.0 | 0.0 | nan}. nan = the report does not say — NOT a zero (rules.md)."""
    t = normalise(text)
    out = {k: _oa_label(t, k) for k in COMPARTMENT}
    for key, pattern in PATTERNS.items():
        if pattern is None:
            continue
        pos = neg = 0
        for m in re.finditer(pattern, t):
            s, e = m.span()
            left, right = t[max(0, s - 45):s], t[e:e + 80]
            negated = (re.search(NEG + r"[^.]{0,25}$", left)
                       or re.search(r"^\W{0,4}(?:" + NEG + r"|ist intakt|intacto|intact)", right))
            if key in NEEDS_ABNORMAL:
                near_abnormal = re.search(ABNORMAL, right[:60]) or re.search(ABNORMAL + r"[^.]{0,30}$", left)
                if not near_abnormal:
                    if re.search(r"^\W{0,6}(?:" + NEG + r"|intact|normal)", right):
                        neg += 1
                    continue
            if negated:
                neg += 1
            else:
                pos += 1
        out[key] = 1.0 if pos else (0.0 if neg else np.nan)
    return out

weak_labels = pd.DataFrame([label_report(r) for r in train["Report"]])[TARGETS]
weak_labels.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values)
print("label coverage (share of studies the report can decide):")
print((weak_labels[TARGETS].notna().mean() * 100).round(1).to_string())


## The model, loaded offline from Kaggle Models


In [ ]:
import torch
LLM_DIRS = ["/kaggle/input/qwen2.5/transformers/7b-instruct/1",
            "/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1",
            "/kaggle/input/qwen-lm/qwen2.5/transformers/7b-instruct/1"]
hits = [d for d in LLM_DIRS if Path(d).exists()]
if not hits:
    found = discover_input("**/config.json")
    hits = [str(p.parent) for p in found if "qwen" in str(p).lower()]
assert hits, f"Qwen2.5-7B-Instruct not attached. Looked in {LLM_DIRS} and searched /kaggle/input."
MODEL_DIR = hits[0]
print("model:", MODEL_DIR)

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
llm = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float16, device_map="auto")
llm.eval()
print("loaded on:", {n: str(p.device) for n, p in list(llm.named_parameters())[:1]})


## The prompt — the whole method, and it lives in `labels/prompt-v1.md`


In [ ]:
PROMPT_VERSION = "own-llm-v1"
SYSTEM = ("You are a musculoskeletal radiologist reading knee MRI reports. Reports may be in "
          "Spanish, English, German, Dutch, Croatian, French, Turkish or other languages. "
          "Answer only with JSON.")
USER_TEMPLATE = """Read this knee MRI report and give the probability that each of the twelve findings is present
IN THIS KNEE, according to the report.

Report:
{report}

Rules:
- Use a probability between 0 and 1 for each finding.
- Use exactly 0.5 when the report DOES NOT ADDRESS that finding. Silence is not a negative:
  0.5 means "the report does not say", not "absent".
- Use a LOW value (0.02-0.10) when the report explicitly states the structure is normal, intact,
  preserved or without tear. A negative statement is information - do not mark it 0.5.
- Use a HIGH value (0.80-0.99) when the report describes the finding as present.
- Osteoarthritis is graded per compartment. "Medial OA", "Lateral OA" and "PF OA" (patellofemoral,
  including retropatellar and trochlear) are separate findings: a statement about one compartment
  says nothing about the others.
- Cartilage loss, chondropathy, chondral thinning, osteophytes and joint-space narrowing all count
  as osteoarthritis in the compartment where they are described.
- "Contusion" means bone marrow oedema, bone bruise or contusion.
- Report your reading of the text only. Do not guess from what is statistically likely in a knee.

Answer with this JSON object and nothing else:
{{"ACL": p, "MCL": p, "Medial Meniscus": p, "Lateral Meniscus": p, "Medial OA": p,
 "Lateral OA": p, "PF OA": p, "Effusion": p, "Synovitis": p, "Baker's": p,
 "Contusion": p, "Fracture": p}}"""

def label_one(report: str) -> tuple[dict, bool]:
    """-> ({finding: probability}, parsed_ok). A failure returns all 0.5 and is counted."""
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": USER_TEMPLATE.format(report=str(report)[:6000])}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok([text], return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**ids, max_new_tokens=220, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        s = gen[gen.index("{"): gen.rindex("}") + 1]
        d = json.loads(s)
        vals = {t: float(np.clip(float(d[t]), 0.0, 1.0)) for t in TARGETS}
        return vals, True
    except Exception:
        return {t: 0.5 for t in TARGETS}, False


## Pilot: label a sample, measure latency, score the prompt on gold-dev / gold-test


In [ ]:
N_PILOT = 300                      # includes all 58 gold studies
rng = np.random.default_rng(12)

gold_uids_all = gold["StudyInstanceUID"].tolist()
perm = rng.permutation(len(gold_uids_all))
DEV = set(np.array(gold_uids_all)[perm[:29]])
TEST = set(np.array(gold_uids_all)[perm[29:]])
print(f"gold split: {len(DEV)} dev / {len(TEST)} test (prompt may only be revised on dev)")

pool = [u for u in train["StudyInstanceUID"] if u not in set(gold_uids_all)]
sample = gold_uids_all + list(rng.choice(pool, max(N_PILOT - len(gold_uids_all), 0), replace=False))
report_of = dict(zip(train["StudyInstanceUID"], train["Report"]))

rows, n_fail, t0 = [], 0, time.time()
for k, uid in enumerate(sample, 1):
    vals, ok = label_one(report_of[uid])
    n_fail += (not ok)
    rows.append({"StudyInstanceUID": uid, **vals})
    if k % 25 == 0 or k == len(sample):
        per = (time.time() - t0) / k
        print(f"  {k}/{len(sample)}  {per:.2f}s/report  "
              f"eta all 4,407: {per*4407/3600:.2f} h  parse failures: {n_fail}", flush=True)

own = pd.DataFrame(rows)
own.to_csv(WORK / f"own_llm_labels_{PROMPT_VERSION}_pilot.csv", index=False)
SEC_PER_REPORT = (time.time() - t0) / len(sample)
print(f"\n{len(own)} reports labelled, {n_fail} parse failures, {SEC_PER_REPORT:.2f} s/report")


In [ ]:
from sklearn.metrics import roc_auc_score

def score(uid_set, key: pd.DataFrame, name: str):
    sel = [u for u in gold_uids_all if u in uid_set]
    y = gold.set_index("StudyInstanceUID").loc[sel, TARGETS].values.astype(int)
    p = key.set_index("StudyInstanceUID").reindex(sel)[TARGETS].values.astype(float)
    p = np.nan_to_num(p, nan=0.5)
    aucs = [roc_auc_score(y[:, j], p[:, j]) for j in range(N_LABEL) if len(np.unique(y[:, j])) > 1]
    per = {t: round(roc_auc_score(y[:, j], p[:, j]), 3) for j, t in enumerate(TARGETS)
           if len(np.unique(y[:, j])) > 1}
    print(f"{name:28} n={len(sel):3d}  macro AUC {np.mean(aucs):.4f}  ({len(aucs)}/12 targets scorable)")
    return float(np.mean(aucs)), per

print(f"prompt {PROMPT_VERSION}, model Qwen2.5-7B-Instruct, greedy\n")
dev_auc, _ = score(DEV, own, "ours — gold-dev")
test_auc, per_t = score(TEST, own, "ours — gold-test")
print()
print(f"reference, same 58 studies: public v4_blend 0.8927 | public v2 0.8873 | our regex 0.6879")
print(f"\nper target (gold-test, n=29):")
print(pd.Series(per_t).sort_values().to_string())

soft = own[TARGETS].values
summary = {"experiment_id": "exp-12-own-llm-labels", "prompt_version": PROMPT_VERSION,
           "model": "Qwen2.5-7B-Instruct", "decoding": "greedy",
           "n_pilot": int(len(own)), "parse_failures": int(n_fail),
           "sec_per_report": round(SEC_PER_REPORT, 2),
           "est_hours_full_corpus": round(SEC_PER_REPORT * 4407 / 3600, 2),
           "gold_dev_auc": round(dev_auc, 4), "gold_test_auc": round(test_auc, 4),
           "share_exactly_0.5": round(float((soft == 0.5).mean()), 3),
           "share_below_0.2": round(float((soft < 0.2).mean()), 3),
           "per_target_gold_test": per_t, "date": time.strftime("%Y-%m-%d")}
with open(WORK / "exp12_pilot_results.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\n" + json.dumps(summary, indent=2)[:900])
budget_check("end of pilot")


## Reading this

- **`share_exactly_0.5`** is the honesty check: the public key sits at 25.4% in v1. Far below that
  means the prompt is guessing where the report is silent; far above means it is refusing to read.
- **`share_below_0.2`** is the fix exp-03 asked for — the regex could barely emit a negative
  (5–8% of decided OA cells). This is where that shows up.
- **`est_hours_full_corpus`** decides whether the full run is affordable before we commit to it.
- **gold-dev vs gold-test** should agree. A large gap means the prompt was tuned to 29 studies.
